In [1]:
# === CELL 0: DIAGNOSTIC CELL ===
import requests
from bs4 import BeautifulSoup
import re

query = "UPI fraud"
page = 1
search_url = f"https://theprint.in/?s={query.replace(' ', '+')}&paged={page}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://theprint.in"
}

print(f"Fetching: {search_url}")
resp = requests.get(search_url, headers=headers, timeout=15)
print(f"HTTP Status: {resp.status_code}")
soup = BeautifulSoup(resp.text, 'html.parser')

cards = soup.find_all('h3', class_=re.compile(r'entry-title', re.I))
print(f"Found {len(cards)} article cards on page {page}")

if cards:
    first = cards[0]
    print(f"\n--- HTML of first card ---")
    print(first.prettify())
    a_tag = first.find('a', href=True)
    if a_tag:
        print(f"\nExtracted Title: {a_tag.get_text(strip=True)}")
        print(f"Extracted URL:   {a_tag.get('href')}")

Fetching: https://theprint.in/?s=UPI+fraud&paged=1
HTTP Status: 200
Found 10 article cards on page 1

--- HTML of first card ---
<h3 class="entry-title td-module-title">
 <a href="https://theprint.in/ani-press-releases/card91-launches-verifyiq-to-strengthen-risk-aligned-onboarding-for-the-bfsi-sector/2893938/" rel="bookmark" title="CARD91 Launches VerifyIQ to Strengthen Risk-Aligned Onboarding for the BFSI Sector">
  CARD91 Launches VerifyIQ to Strengthen Risk-Aligned Onboarding for the BFSI Sector
 </a>
</h3>


Extracted Title: CARD91 Launches VerifyIQ to Strengthen Risk-Aligned Onboarding for the BFSI Sector
Extracted URL:   https://theprint.in/ani-press-releases/card91-launches-verifyiq-to-strengthen-risk-aligned-onboarding-for-the-bfsi-sector/2893938/


In [2]:
# === CELL 1: IMPORTS AND CONFIGURATION ===
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
import random
import os
import re
from datetime import datetime

# ── ID Continuity ──────────────────────────────────────────────────────────
existing_max_id = 0
for csv_path in [
    "data/consumer_complaints.csv",
    "data/indiankanoon_complaints.csv",
    "data/medianama_complaints.csv",
    "data/reddit_complaints.csv",
    "data/inc42_complaints.csv",
    "data/entrackr_complaints.csv",
]:
    if os.path.exists(csv_path):
        df_check = pd.read_csv(csv_path)
        if "Unique ID" in df_check.columns:
            ids = df_check["Unique ID"].dropna().str.extract(r'(\d+)')[0].astype(float)
            if not ids.empty:
                existing_max_id = max(existing_max_id, int(ids.max()))

for fname in os.listdir("data/txt"):
    match = re.search(r'NA-(\d+)\.txt', fname)
    if match:
        existing_max_id = max(existing_max_id, int(match.group(1)))

next_id = existing_max_id + 1
print(f"Starting Unique ID: NA-{next_id:04d}")

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_URL = "https://theprint.in"
OUTPUT_CSV = "data/theprint_complaints.csv"
OUTPUT_JSON = "data/theprint.json"
TXT_DIR = "data/txt"
os.makedirs(TXT_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)
MAX_PAGES = 5

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://theprint.in"
}

if 'session' not in globals():
    session = requests.Session()

Starting Unique ID: NA-28130


In [3]:
# === CELL 2: KEYWORD TAXONOMY ===
KEYWORD_TAXONOMY = {
    "General Cybercrime / Cyber Fraud Terms": {
        "General Cybercrime": ["cyber crime", "cybercrime", "cyber fraud", "online fraud", "internet fraud", "digital fraud"],
        "Cyber Scam": ["cyber scam", "online scam", "internet scam", "online cheating"],
        "Financial Cyber Fraud": ["financial fraud online", "net banking fraud", "e-banking fraud"]
    },
    "UPI and Digital Payment Fraud": {
        "UPI Fraud": ["UPI fraud", "UPI scam", "Google Pay fraud", "PhonePe fraud", "Paytm fraud", "BHIM fraud"],
        "QR Code Fraud": ["QR code scam", "QR code fraud", "scan and pay fraud"],
        "Payment Link Fraud": ["payment link fraud", "collect request scam", "fake payment link"],
        "Mobile Wallet Fraud": ["mobile wallet fraud", "e-wallet scam", "digital wallet fraud"]
    },
    "OTP and Authentication Fraud": {
        "OTP Fraud": ["OTP fraud", "OTP scam", "OTP theft"],
        "SIM Swap": ["SIM swap fraud", "SIM cloning", "duplicate SIM fraud"],
        "KYC Fraud": ["KYC fraud", "KYC scam", "Aadhaar KYC scam"]
    },
    "Digital Arrest Scam": {
        "Digital Arrest": ["digital arrest", "digital arrest scam", "fake arrest", "video call arrest"],
        "Impersonation Scam": ["police impersonation scam", "CBI fraud call", "customs fraud call", "TRAI scam call", "ED scam call"],
        "Video Call Coercion": ["video call scam", "video call blackmail", "video call extortion", "fake interrogation"]
    },
    "Phishing, Vishing, and Smishing": {
        "Phishing": ["phishing", "phishing attack", "phishing email", "fake website", "spoof website"],
        "Vishing": ["vishing", "voice phishing", "fraud call", "fake bank call"],
        "Smishing": ["smishing", "SMS fraud", "SMS scam", "phishing SMS"]
    },
    "Online Lending and Loan App Fraud": {
        "Loan App Fraud": ["loan app fraud", "instant loan scam", "loan app harassment", "illegal loan app"],
        "Loan App Extortion": ["loan app blackmail", "loan app threat", "morphed photos loan", "recovery agent threat"]
    },
    "Investment and Trading Fraud": {
        "Investment Scam": ["investment scam", "Ponzi scheme", "online investment fraud", "crypto scam", "bitcoin fraud", "forex trading scam", "pig butchering"],
        "Stock Market Fraud": ["stock market scam", "share trading fraud", "demat fraud", "pump and dump"],
        "Task Scam": ["task fraud", "part time job scam", "work from home scam", "Telegram task scam"]
    },
    "Identity Theft and Data Breach": {
        "Identity Theft": ["identity theft", "identity fraud", "Aadhaar misuse", "PAN fraud"],
        "Data Breach": ["data breach", "data leak", "data theft", "customer data breach"]
    },
    "Social Engineering and Romance/Sextortion": {
        "Social Engineering": ["social engineering fraud", "manipulation scam", "trust scam"],
        "Romance Scam": ["romance scam", "dating fraud", "matrimonial fraud", "honey trap", "catfishing fraud"],
        "Sextortion": ["sextortion", "webcam blackmail", "nude video blackmail"]
    },
    "E-Commerce and Delivery Fraud": {
        "E-Commerce Fraud": ["e-commerce fraud", "online shopping fraud", "fake product scam", "Flipkart fraud", "Amazon fraud", "refund scam"],
        "Delivery Fraud": ["fake delivery", "courier fraud", "customs duty scam", "parcel scam", "delivery OTP scam"]
    },
    "Ransomware and Malware": {
        "Ransomware": ["ransomware attack", "cyber ransom", "data encryption attack"],
        "Banking Malware": ["banking trojan", "banking malware", "keylogger fraud", "AnyDesk fraud", "TeamViewer scam", "screen sharing scam"]
    },
    "Emerging and Miscellaneous Fraud Types": {
        "Deepfake Fraud": ["deepfake scam", "deepfake fraud", "AI voice scam", "voice cloning scam"],
        "Utility Scam": ["electricity bill scam", "utility fraud", "disconnection scam"],
        "Aadhaar Fraud": ["Aadhaar fraud", "Aadhaar scam", "biometric fraud", "AEPS fraud"],
        "Cyber Stalking": ["cyber stalking", "cyber bullying", "online harassment", "digital harassment"]
    }
}

In [4]:
# === CELL 3: HELPER FUNCTIONS ===
def classify_narrative_type(text):
    text_lower = text.lower()
    if any(x in text_lower for x in ["victim", "lost", "cheated", "defrauded",
                                       "money stolen", "account hacked", "fell for"]):
        return "VICTIM"
    elif any(x in text_lower for x in ["beware", "warning", "alert", "avoid",
                                         "do not", "scam alert", "red flag"]):
        return "NEAR-MISS"
    return "THIRD-PARTY"

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def safe_save(df, csv_path, json_path, json_data):
    try:
        temp_csv = csv_path + ".tmp"
        df.to_csv(temp_csv, index=False, encoding="utf-8-sig")
        os.replace(temp_csv, csv_path)
        temp_json = json_path + ".tmp"
        with open(temp_json, "w", encoding="utf-8") as f:
            json.dump(json_data, f, indent=4, ensure_ascii=False)
        os.replace(temp_json, json_path)
        print(f"  ✅ Checkpoint saved: {len(df)} records")
    except Exception as e:
        print(f"  ⚠️ Checkpoint save failed (data still in memory): {e}")

In [5]:
# === CELL 4: SEARCH FUNCTION ===
def search_theprint(query, page=1):
    """Search ThePrint and return list of result dicts with title, url, date."""
    results = []
    seen_urls = set()
    search_url = f"https://theprint.in/?s={query.replace(' ', '+')}&paged={page}"

    try:
        time.sleep(random.uniform(2.0, 4.0))
        response = session.get(search_url, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            print(f"  HTTP {response.status_code} for page {page}")
            return results

        soup = BeautifulSoup(response.text, 'html.parser')

        # Primary selector: h3 tags with entry-title class
        cards = soup.find_all('h3', class_=re.compile(r'entry-title', re.I))

        # Fallback: h2 tags
        if not cards:
            cards = soup.find_all('h2', class_=re.compile(r'entry-title', re.I))

        if not cards:
            print(f"  No article cards found on page {page}")
            return results

        for card in cards:
            a_tag = card.find('a', href=True)
            if not a_tag:
                continue
            url = a_tag.get('href', '')
            title = a_tag.get_text(strip=True)

            # Skip press releases — low quality aggregated content
            if '/ani-press-releases/' in url:
                continue
            if not url.startswith('https://theprint.in/'):
                continue
            if url in seen_urls or len(title) < 10:
                continue
            seen_urls.add(url)

            # Try to get date from nearby time tag
            parent = card.find_parent(['div', 'li', 'article'])
            time_tag = parent.find('time') if parent else None
            date = time_tag.get('datetime', '')[:10] if time_tag and time_tag.get('datetime') else "Unknown Date"

            results.append({"title": title, "url": url, "date": date})

        print(f"  Page {page}: found {len(results)} results")
        return results

    except Exception as e:
        print(f"  Error on page {page} for '{query}': {e}")
        return results

In [6]:
# === CELL 5: ARTICLE FETCH FUNCTION ===
def fetch_theprint_article(url):
    """Fetch full text, author, and date from a ThePrint article page."""
    try:
        time.sleep(random.uniform(2.0, 3.5))
        response = session.get(url, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            return "", "Unknown", "Unknown Date"

        soup = BeautifulSoup(response.text, 'html.parser')

        # Remove noise
        for tag in soup.find_all(['script', 'style', 'aside', 'nav', 'figure', 'iframe']):
            tag.decompose()
        for cls in ['related-posts', 'social-share', 'newsletter', 'sidebar', 'comments', 'advertisement']:
            for el in soup.find_all(class_=re.compile(cls, re.I)):
                el.decompose()

        # Content
        content = (
            soup.find('div', class_=re.compile(r'td-post-content', re.I)) or
            soup.find('div', class_='entry-content') or
            soup.find('article') or
            soup.find('main')
        )
        text = clean_text(content.get_text(separator='\n', strip=True)) if content else ""

        # Date — first <time datetime="..."> on the page
        time_tag = soup.find('time')
        date = time_tag.get('datetime', '')[:10] if time_tag and time_tag.get('datetime') else "Unknown Date"

        # Author
        author_tag = (
            soup.find('a', rel='author') or
            soup.find(class_=re.compile(r'td-post-author-name|author-name|byline', re.I))
        )
        author = author_tag.get_text(strip=True) if author_tag else "The Print"

        return text, author, date

    except Exception as e:
        print(f"  Error fetching {url}: {e}")
        return "", "Unknown", "Unknown Date"

In [7]:
# === CELL 6: MAIN SCRAPING LOOP ===
existing_urls = set()
existing_data = []
if os.path.exists(OUTPUT_JSON):
    try:
        with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
            existing_data = json.load(f)
        for item in existing_data:
            if item.get("URL"):
                existing_urls.add(item["URL"])
    except json.JSONDecodeError:
        pass
print(f"Loaded {len(existing_urls)} existing URLs to skip")

today_date = datetime.now().strftime("%Y-%m-%d")
all_new_records = []
all_new_raw = []
new_count = 0

for parent_category, subcategories in KEYWORD_TAXONOMY.items():
    for subcat, keywords in subcategories.items():
        for keyword in keywords:
            print(f"\n[*] '{keyword}' | {parent_category} > {subcat}")
            for page_num in range(1, MAX_PAGES + 1):
                search_results = search_theprint(keyword, page=page_num)
                if not search_results:
                    break
                for res in search_results:
                    url = res['url']
                    if url in existing_urls:
                        continue
                    existing_urls.add(url)
                    print(f"  Fetching: {url[:80]}")
                    full_text, author, date = fetch_theprint_article(url)
                    if date == "Unknown Date" and res['date'] != "Unknown Date":
                        date = res['date']
                    title = res['title']
                    unique_id = f"NA-{next_id:04d}"
                    txt_filename = f"{unique_id}.txt"
                    txt_content  = f"SOURCE: The Print\n"
                    txt_content += f"TITLE: {title}\n"
                    txt_content += f"AUTHOR: {author}\n"
                    txt_content += f"DATE: {date}\n"
                    txt_content += f"URL: {url}\n\n"
                    txt_content += "--- ARTICLE TEXT ---\n\n"
                    txt_content += full_text if full_text else "[Article text unavailable]"
                    with open(os.path.join(TXT_DIR, txt_filename), "w", encoding="utf-8") as f:
                        f.write(txt_content)
                    record = {
                        "Unique ID": unique_id,
                        "Date of Collection": today_date,
                        "Collector Name": "Soubhik Sarkar",
                        "Source Platform": "The Print",
                        "Source Publication": "theprint.in",
                        "Original Date": date,
                        "Title/Headline": title,
                        "URL": url,
                        "Search Query Used": keyword,
                        "Fraud Category": parent_category,
                        "Fraud Subcategory": subcat,
                        "Narrative Type": classify_narrative_type(full_text),
                        "TXT File Name": txt_filename,
                        "Notes": f"Author: {author}"
                    }
                    raw_item = {
                        "URL": url,
                        "Title/Headline": title,
                        "Original Date": date,
                        "Author": author,
                        "StructuredData": record
                    }
                    all_new_records.append(record)
                    all_new_raw.append(raw_item)
                    next_id += 1
                    new_count += 1
                    if new_count % 25 == 0:
                        df_temp = pd.DataFrame(all_new_records)
                        combined_raw = existing_data + all_new_raw
                        safe_save(df_temp, OUTPUT_CSV, OUTPUT_JSON, combined_raw)

print(f"\nDone. Total new records this session: {new_count}")

Loaded 0 existing URLs to skip

[*] 'cyber crime' | General Cybercrime / Cyber Fraud Terms > General Cybercrime
  Page 1: found 10 results
  Fetching: https://theprint.in/india/rajasthan-dgp-reviews-cyber-crime-branch-orders-expans
  Fetching: https://theprint.in/india/bengal-police-arrest-industrialist-pawan-ruia-in-multi
  Fetching: https://theprint.in/india/cctns-2-0-to-be-ai-powered-crime-prediction-criminal-p
  Fetching: https://theprint.in/india/rajkot-cop-female-friend-injured-after-his-service-wea
  Fetching: https://theprint.in/india/zpm-womens-wing-files-police-complaint-over-defamatory
  Fetching: https://theprint.in/india/delhi-tops-cybersecurity-incidents-as-cases-double-nat
  Fetching: https://theprint.in/india/punjab-retired-itbp-officer-loses-rs-17-52-lakh-in-onl
  Fetching: https://theprint.in/india/man-from-bihar-held-for-online-fraud-in-jkhand/2888387
  Fetching: https://theprint.in/india/parliament-budget-session-resumes-today-modi-to-addres
  Fetching: https://thep

In [ ]:
# === CELL 7: FINAL SAVE ===
if all_new_records:
    df_new = pd.DataFrame(all_new_records)

    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        df_combined.drop_duplicates(subset=["URL"], keep="last", inplace=True)
    else:
        df_combined = df_new

    temp_csv = OUTPUT_CSV + ".tmp"
    df_combined.to_csv(temp_csv, index=False, encoding="utf-8-sig")
    os.replace(temp_csv, OUTPUT_CSV)
    df_combined.to_excel(OUTPUT_CSV.replace(".csv", ".xlsx"), index=False)

    combined_raw = existing_data + all_new_raw
    json_dedup = {v["URL"]: v for v in combined_raw}
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(list(json_dedup.values()), f, indent=4, ensure_ascii=False)

    ids = df_combined["Unique ID"].dropna().str.extract(r'(\d+)')[0].astype(float)
    print(f"✅ Saved {len(df_combined)} total records")
    if not ids.empty:
        print(f"IDs: NA-{int(ids.min()):04d} to NA-{int(ids.max()):04d}")
    print(f"CSV: {OUTPUT_CSV}")
    print(f"JSON: {OUTPUT_JSON}")
    print(f"TXT files: {TXT_DIR}/")
else:
    print("No new records to save.")